In [ ]:
%load_ext autoreload
%autoreload 2

# Setup Environment

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.globals import set_debug, set_verbose
set_debug(False)
set_verbose(False)

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [ ]:
lab_tech_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature = 0.0,
    top_p=0.95
)

triage_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature = 0.3,
    top_p=0.95
)

diag_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature = 0.7,
    top_p=0.95
)

# PROMPT

In [ ]:
LAB_TECHNICIAN = """
You are a Lab Technician. Your role is to consolidate all available patient and image data into a single, structured report. You will perform an objective visual analysis of the provided image, but you MUST NOT make a diagnosis or infer any disease. Your task is to report only the facts.

Instructions:
  1. Receive the patient's metadata (age, sex, location, melanocytic status).
  2. Receive the dermatoscopic image.
  3. For the `dermoscopic_features` section, identify ALL relevant features you observe in the image. For each feature, create a separate object in the list containing its name and a detailed description.
  4. Include common features like pigment patterns, vascular structures, and surface characteristics. If a common feature (like a pigment network) is absent, do not create an object for it.

Output Format:
You must output ONLY a single, valid JSON object with the following structure.
{
  "patient_data": {
    "age": <age>,
    "sex": "<sex>",
    "lesion_location": "<location>"
  },
  "lesion_metadata": {
    "is_melanocytic": <true | false>
  },
  "visual_summary": {
    "symmetry": "<Symmetrical | Asymmetrical>",
    "colors": ["<color1>", "<color2>", "..."],
    "border_characteristics": "<Well-defined | Ill-defined | Irregular | Fading>"
  },
  "dermoscopic_features": [
    {
      "feature_name": "<The name of the first observed feature, e.g., 'Pigment Globules'>",
      "description": "<A detailed description of this feature, including its color, size, and location.>"
    },
    {
      "feature_name": "<The name of the second observed feature, e.g., 'Surface Texture'>",
      "description": "<A detailed description of the surface and texture, e.g., 'The surface appears smooth and slightly waxy in the center, with some fine scaling at the periphery.'>"
    },
    {
      "feature_name": "<The name of the third observed feature, e.g., 'Vascular Pattern'>",
      "description": "<A detailed description of the vessels, e.g., 'Fine linear vessels are visible in the reddish areas, consistent with telangiectasias.'>"
    }
  ]
}
"""

TRIAGE_PROMPT = """
You are an expert AI Triage Specialist. Your task is to determine the broad family of a skin lesion. You will receive an initial report containing metadata AND the original image.

Instructions:
1.  First, check the `is_melanocytic` flag in the report. This is your primary guide.
2.  IF `is_melanocytic` is `true`, classify the family as "Melanocytic Proliferations".
3.  IF `is_melanocytic` is `false`, you must analyze the provided image directly, using the `visual_summary` as context, to classify the lesion into one of the following NON-melanocytic families:
    - "Keratinocyte Tumors (Benign)"
    - "Keratinocyte Tumors (Malignant/Pre-malignant)"
    - "Fibrohistiocytic Tumors"
4.  Your visual analysis should confirm the classification. For example, if you see features of a BCC in the image, that strongly supports the 'Malignant' category.

Output Format:
{
  "disease_family": "<The name of the classified disease family>",
  "reasoning": "<Your reasoning for the classification>"
}
"""

DX_PROMPT = """
You are a specialist AI Dermoscopist. Your task is to provide a final, specific diagnosis by performing a direct visual comparison between a query image and a set of confirmed reference cases.

INPUTS YOU WILL RECEIVE:
-  The original `query_image`.
-  An `initial_report` (containing patient data and a visual summary for context).
-  A `disease_family` classification from the Triage Agent.
-  A list of `reference_cases`, where each case is an image with its confirmed diagnosis from within the assigned family.

INSTRUCTIONS:
1. Analyze Query Image: Briefly describe the key visual features of the `query_image` to establish a baseline for comparison.
2. Perform Comparative Analysis: Perform a head-to-head visual comparison of the `query_image` against EACH of the provided `reference_cases`. For each comparison, you must justify the visual similarities and differences.
3. Synthesize and Decide: Based on your comparative analysis, you must determine:
  - The single best visual match. The diagnosis of this case will be your `final_diagnosis`.
  - The single next-best visual match. The diagnosis of this case will be your `differential_diagnosis`.
4. Set Confidence: Your confidence level should reflect the strength of the best match. If the best match is visually very similar and clearly superior to the others, confidence is 'High'. If the best match is only a fair match, or if it is difficult to distinguish between the top two matches, confidence MUST be 'Medium' or 'Low'.

OUTPUT FORMAT:
Output ONLY a single JSON object with your final report. The reasoning must detail your comparative analysis.
{
  "final_diagnosis": "<The final diagnosis>",
  "confidence": "Low" | "Medium" | "High",
  "differential_diagnosis": "<The diagnosis of the second best-matching reference case>",
  "reasoning": {
    "query_summary": "<Your brief visual feature description of the query image.>",
    "comparative_analysis": [
      {
        "reference_diagnosis": "<Diagnosis of Reference Case 1>",
        "justification": "<Your concise detailed reasoning for the visual match, highlighting specific similar or different features.>"
      },
      {
        "reference_diagnosis": "<Diagnosis of Reference Case 2>",
        "justification": "<Your concise detailed reasoning for the visual match...>"
      }
    ],
    "final_synthesis": "<Your concise final conclusion explaining why the chosen reference case is a better visual match than the differential diagnosis.>"
  }
}
"""

# Ensure custom import module

In [ ]:
import sys
from pathlib import Path

root_dir = Path.cwd().parent 

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

print("Added project root to path:", root_dir)

# Agent Nodes

In [ ]:
from typing import TypedDict, Dict, List, Any
from utils import add_trace, encode_image, invoke_llm, run_evaluation_in_batch
from retrieve_image import QdrantBiomedCLIPRetriever

import json

COLLECTION_NAME = "derm_vkb"
QDRANT_URL = "http://localhost:6333"

class DermState(TypedDict):
    patient_data: Dict[str, Any]
    image_path: str
    lab_report: Dict[str, Any]
    triage_result: Dict[str, Any]
    diagnose_result: Dict[str, Any]
    trace: List[Dict[str, Any]]
    
def lab_technician(state: DermState):
    image_path = state.get("image_path", "")
    image_data = encode_image(image_path)
    patient_data = state.get("patient_data", {})
    
    user_message = [
        {
            "type": "text",
            "text": "Process the following case and generate the initial report.\n"
        },
        {
            "type": "text",
            "text": f"Patient Data:\n {patient_data}\n"
        },
        {
            "type": "image_url",
            "image_url": { "url": f"data:image/jpeg;base64,{image_data}" }
        }
    ]
    
    try:
        lab_report = invoke_llm(lab_tech_llm, LAB_TECHNICIAN, user_message)  
        add_trace(state, agent="lab_technician", role="user", payload=lab_report)

    except Exception as e:
        print(f"Error occurred during feature extraction: {e}")
    
    return {
        "lab_report": lab_report
    }
    
def triage_node(state: DermState):
    lab_report = state.get("lab_report", {})
    image_data = encode_image(state.get("image_path", ""))
    
    user_message = [
        {
            "type": "text",
            "text": "Based on the following report AND the provided image, perform triage and classify the disease family.\n"
        },
        {
            "type": "text",
            "text": "Skin Lesion Image:\n"
        },
        {
            "type": "image_url",
            "image_url": { "url": f"data:image/jpeg;base64,{image_data}" }
        },
        {
            "type": "text",
            "text": f"Lab Report:\n{json.dumps(lab_report, ensure_ascii=False)}"
        },
    ]

    triage_result = invoke_llm(triage_llm, TRIAGE_PROMPT, user_message)
    add_trace(state, agent="triage", role="user", payload=triage_result)

    return {
        "triage_result": triage_result
    }
    
def diagnosis_node(state: DermState):
    lab_report = state.get("lab_report", {})
    image_path = state.get("image_path", "")
    image_data = encode_image(image_path)
    
    user_message = [
        {
            "type": "text",
            "text": "Based on the following report AND the provided image, perform triage and classify the disease family.\n"
        },
        {
            "type": "text",
            "text": "Skin Lesion Image:\n"
        },
        {
            "type": "image_url",
            "image_url": { "url": f"data:image/jpeg;base64,{image_data}" }
        },
        {
            "type": "text",
            "text": f"Lab Report:\n{json.dumps(lab_report, ensure_ascii=False)}"
        },
    ]
    
    # Retrieved similar cases
    retriever = QdrantBiomedCLIPRetriever(collection=COLLECTION_NAME, qdrant_url=QDRANT_URL)
    results = retriever.search(image_path=image_path, k=3)
    
    if results:
        for i, result in enumerate(results):
            case_image = encode_image(result.get("image_path", ""))
            user_message.append({
                "type": "text",
                "text": f"Reference Case #{i + 1}: Diagnosis: {result['diagnosis']}, age: {result['age']}, sex: {result['sex']}, lesion_site: {result['anatom_site']}\n"
            })
            user_message.append({
                "type": "image_url",
                "image_url": { "url": f"data:image/jpeg;base64,{case_image}" }
            })

    dx_result = invoke_llm(diag_llm, DX_PROMPT, user_message)
    add_trace(state, agent="diagnose", role="user", payload=dx_result)

    return {
        "diagnose_result": dx_result
    }

# LANGGRAPH FLOW

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver

workflow = StateGraph(DermState)

workflow.add_node("lab_technician", lab_technician)
workflow.add_node("triage", triage_node)
workflow.add_node("diagnosis", diagnosis_node)

workflow.set_entry_point("lab_technician")

workflow.add_edge("lab_technician", "triage")
workflow.add_edge("triage", "diagnosis")
workflow.add_edge("diagnosis", END)

checkpointer = InMemorySaver()

app = workflow.compile(checkpointer=checkpointer)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    pass

# INFERENCE

In [ ]:
import pickle as pkl

with open("../dataset/300_test_set.pkl", "rb") as f:
    test_set = pkl.load(f)
    
image_path = "../dataset/test"
for item in test_set:
    file_name = Path(item["image_path"]).name
    item["image_path"] = f"{image_path}/{file_name}"

In [ ]:
OUTPUT_DIR = "../results"
OUT_FILE = "gpt-5-mini.jsonl"

jsonl_file = run_evaluation_in_batch(app, test_set, out_dir=OUTPUT_DIR, out_file=OUT_FILE, resume=True)
print("JSONL at:", jsonl_file)